# Qwen-27B Semantic Error Control and Code Mediation

This confirmatory audit freezes the Qwen-27B L49 physical-side response site and L55 answer-code site discovered in the preceding depth profile.

[实验设置与运行说明](../../../docs/experiment_notes/11_handoff_development_and_matching/ca64541fd11581ce.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 17 tracked cells.')


## 1. Setup


In [ ]:
%%cell_progress 1/17 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/17 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'pandas>=2.2', 'matplotlib>=3.8', 'scikit-learn>=1.5',
])

cell_stage('Registering the live VisualTorque source tree')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
assert (PACKAGE_ROOT / '__init__.py').is_file(), PACKAGE_ROOT
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if (
        module_name == 'visual_torque_toolkit'
        or module_name.startswith('visual_torque_toolkit.')
    ):
        del sys.modules[module_name]
importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
import visual_torque_toolkit
from IPython.display import display
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter

required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name)
    for name in required_features
)
toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents, toolkit_path
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit source:', toolkit_path)


In [ ]:
%%cell_progress 3/17 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
print(torch.cuda.get_device_name(0))
print('CUDA memory (GB):', round(
    torch.cuda.get_device_properties(0).total_memory / 2**30, 1
))

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Public checkpoints may still work.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/17 Lock the semantic-error mediation protocol
cell_stage('Freezing model, sites, supports, mappings, cohorts, and gates')
import hashlib
import json
from datetime import datetime, timezone

MODEL_KEY = 'qwen27b'
MODEL_ID = 'Qwen/Qwen3.5-27B'
MODEL_SLUG = 'Qwen__Qwen3.5-27B__373612cb'
MODEL_REVISION = '373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
EXPECTED_LAYER_COUNT = 64
SEMANTIC_LAYER = 49
LABEL_LAYER = 55
CAPTURE_LAYERS = (SEMANTIC_LAYER, LABEL_LAYER)
SITE_SCOPE = 'final_prompt_token'

SEED = 20260902
DIRECTION_SEED = 20260901
LABELS = ('A', 'B', 'C')
MAPPING_VARIANTS = ('forward', 'reverse')
DONOR_TASKS = ('weight_more', 'weight_less')
CONFLICT_CELLS = ('weight_conflict', 'distance_conflict')
EXPECTED_DIRECTION_FAMILIES = 32
EXPECTED_TARGET_FAMILIES = 48
INTERVENTIONS = (
    'l49_semantic_exact', 'l49_semantic_shortcut',
    'l49_random_exact', 'l49_random_shortcut',
    'l55_code_exact', 'l55_code_shortcut',
    'l55_stale_exact', 'l55_stale_shortcut',
    'l55_random_exact', 'l55_random_shortcut',
)
RECORDS_PER_VARIANT = len(INTERVENTIONS)
RECORDS_PER_FAMILY = len(MAPPING_VARIANTS) * RECORDS_PER_VARIANT
EXPECTED_RECORDS = EXPECTED_TARGET_FAMILIES * RECORDS_PER_FAMILY
PATCH_CONDITION_BATCH_SIZE = 1
BOOTSTRAP_REPLICATES = 10000
ALPHA = 0.05
MIN_COHORT = 8
ERROR_SESOI_LOGODDS = 0.25
MEDIATION_SESOI_COSINE = 0.02
SMOKE_CENTERED_LOGPROB_TOLERANCE = 0.15
NORM_RELATIVE_TOLERANCE = 0.02
IMAGE_TOKEN_ID_OVERRIDE = None

DEPTH_ROOT = PROJECT_ROOT / 'results' / 'semantic_label_depth_localization_v1'
DEPTH_SUPPORT_PATH = DEPTH_ROOT / 'depth_support.jsonl'
DEPTH_DECISION_PATH = (
    DEPTH_ROOT / 'models' / MODEL_SLUG / 'analysis' / 'depth_decision.json'
)
INVERSE_SCENE_DIR = (
    PROJECT_ROOT / 'results' / 'inverse_query_weight_route_specificity_v1'
    / 'rendered_scenes'
)
TARGET_ROOT = (
    PROJECT_ROOT / 'results' / 'targeted_shortcut_error_rescue_induction_v1'
)
TARGET_MANIFEST_PATH = TARGET_ROOT / 'shortcut_error_manifest.jsonl'
TARGET_SCENE_DIR = TARGET_ROOT / 'rendered_scenes'
TARGET_PRIOR_DECISION_PATH = (
    TARGET_ROOT / 'models' / MODEL_SLUG / 'analysis'
    / 'targeted_error_decision.json'
)

SHARED_ROOT = (
    PROJECT_ROOT / 'results' / 'qwen27b_semantic_error_mediation_v1'
)
PROTOCOL_LOCK_PATH = SHARED_ROOT / 'frozen_protocol.json'
SUPPORT_LOCK_PATH = SHARED_ROOT / 'support_lock.json'
SHARD_DIR = SHARED_ROOT / 'target_shards'
ANALYSIS_DIR = SHARED_ROOT / 'analysis'
FIGURE_DIR = ANALYSIS_DIR / 'figures'
for path in (SHARED_ROOT, SHARD_DIR, ANALYSIS_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

protocol_identity = {
    'schema_version': 'qwen27b-semantic-error-mediation-protocol-1.0',
    'analysis_status': 'confirmatory_on_preexisting_independent_conflicts',
    'model_id': MODEL_ID,
    'revision': MODEL_REVISION,
    'semantic_site': [SEMANTIC_LAYER, SITE_SCOPE],
    'label_site': [LABEL_LAYER, SITE_SCOPE],
    'site_selection': 'frozen from semantic-label depth localization',
    'direction_support': '32 locked depth-localization families only',
    'target_support': '48 pre-existing shortcut-conflict families only',
    'target_selection_uses_current_outputs': False,
    'mapping_variants': list(MAPPING_VARIANTS),
    'mapping_rule': (
        'forward and reverse swap the labels assigned to exact and shortcut '
        'semantics while preserving the same image and balanced option'
    ),
    'stable_rescue_cohort': (
        'both clean mappings predict the specified shortcut semantic'
    ),
    'stable_induction_cohort': (
        'both clean mappings predict the exact-torque semantic'
    ),
    'primary_error_pool': 'weight and distance conflict families pooled',
    'error_gate': (
        'cohort n >= 8; main lower 95% CI > 0.25; random-subtracted '
        'lower CI > 0; both mapping-specific lower CIs > 0; BH q < 0.05'
    ),
    'mediation_gate': (
        'for exact and shortcut semantic steering, downstream L55 cosine '
        'alignment with the target-specific code exceeds the stale-mapping '
        'code by lower 95% CI > 0.02 and BH q < 0.05'
    ),
    'code_control_gate': (
        'target-specific L55 code steering exceeds stale-mapping code '
        'steering and equal-norm random steering for exact and shortcut '
        'directions; lower 95% CIs > 0 and BH q < 0.05'
    ),
    'random_control': 'deterministic equal-norm direction at each site',
    'bootstrap_unit': 'independent physical target family',
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': SEED,
    'interpretation_limit': (
        'A positive result establishes causal control by an abstract response '
        'semantic and its conversion to output code; it does not prove that '
        'the semantic was computed by exact torque rather than an upstream cue.'
    ),
}
canonical_protocol_identity = json.loads(json.dumps(
    protocol_identity, sort_keys=True
))
if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == canonical_protocol_identity, (
        'Existing protocol differs. Use a new versioned output root.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': canonical_protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

print('Model:', MODEL_ID, '@', MODEL_REVISION)
print('Frozen semantic / code sites:', SEMANTIC_LAYER, LABEL_LAYER)
print('Direction / target families:',
      EXPECTED_DIRECTION_FAMILIES, EXPECTED_TARGET_FAMILIES)
print('Expected intervention records:', EXPECTED_RECORDS)
print('Output:', SHARED_ROOT)


In [ ]:
%%cell_progress 5/17 Verify frozen discoveries and prior supports
cell_stage('Checking L49 semantic, L55 label, and the locked conflict manifest')

for required in (
    DEPTH_SUPPORT_PATH, DEPTH_DECISION_PATH, INVERSE_SCENE_DIR,
    TARGET_MANIFEST_PATH, TARGET_SCENE_DIR, TARGET_PRIOR_DECISION_PATH,
):
    assert Path(required).exists(), f'Missing prerequisite: {required}'

depth_decision = json.loads(DEPTH_DECISION_PATH.read_text(encoding='utf-8'))
assert depth_decision['model_id'] == MODEL_ID
assert depth_decision['resolved_revision'] == MODEL_REVISION
assert depth_decision['transition_decision'] == 'semantic_to_label_transition'
assert SEMANTIC_LAYER in depth_decision['semantic_layers']
assert int(depth_decision['stable_label_onset_layer']) == LABEL_LAYER

prior_error_decision = json.loads(
    TARGET_PRIOR_DECISION_PATH.read_text(encoding='utf-8')
)
assert prior_error_decision['model_id'] == MODEL_ID
assert int(prior_error_decision['family_count']) == EXPECTED_TARGET_FAMILIES
print('Frozen depth decision:', depth_decision['transition_decision'])
print('Prior conflict decision:', prior_error_decision['primary_decision'])


## 2. Disjoint direction and independent conflict supports


In [ ]:
%%cell_progress 6/17 Lock disjoint direction and target supports
cell_stage('Auditing geometry disjointness and caching both image sets')
from collections import Counter, defaultdict
import shutil

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def stable_key(value, seed=SEED):
    return hashlib.sha256(f'{seed}:{value!r}'.encode('utf-8')).hexdigest()

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def canonical_signature(stacks):
    return tuple(sorted(
        (str(stack['side']), int(stack['distance']), int(stack['units']))
        for stack in stacks
    ))

direction_support = read_jsonl(DEPTH_SUPPORT_PATH)
target_manifest = read_jsonl(TARGET_MANIFEST_PATH)
assert len(direction_support) == EXPECTED_DIRECTION_FAMILIES
assert len(target_manifest) == EXPECTED_TARGET_FAMILIES
assert Counter(row['diagnostic_cell'] for row in target_manifest) == {
    'weight_conflict': 24, 'distance_conflict': 24,
}

direction_signatures = {
    canonical_signature(role['stacks'])
    for row in direction_support for role in row['roles'].values()
}
target_signatures = [
    canonical_signature(row['roles']['base']['stacks'])
    for row in target_manifest
]
assert len(set(target_signatures)) == len(target_signatures)
assert not (set(target_signatures) & direction_signatures)

physics_violations = []
for row in target_manifest:
    base = row['roles']['base']['features']
    exact = str(base['outcome'])
    shortcut = str(base['strategy_predictions'][
        row['shortcut_type'] + '_only'
    ])
    if exact not in ('left', 'right') or shortcut not in ('left', 'right'):
        physics_violations.append((row['family_id'], 'nonbinary'))
    if exact == shortcut:
        physics_violations.append((row['family_id'], 'no_conflict'))
assert not physics_violations

support_identity = {
    'schema_version': 'qwen27b-semantic-error-support-1.0',
    'depth_support_sha256': sha256_file(DEPTH_SUPPORT_PATH),
    'depth_decision_identity': {
        'transition_decision': depth_decision['transition_decision'],
        'semantic_layers': depth_decision['semantic_layers'],
        'stable_label_onset_layer': depth_decision['stable_label_onset_layer'],
        'resolved_revision': depth_decision['resolved_revision'],
    },
    'target_manifest_sha256': sha256_file(TARGET_MANIFEST_PATH),
    'direction_family_ids': sorted(row['family_id'] for row in direction_support),
    'target_family_ids': sorted(row['family_id'] for row in target_manifest),
    'geometrically_disjoint': True,
}
if SUPPORT_LOCK_PATH.is_file():
    previous = json.loads(SUPPORT_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == support_identity
else:
    write_json_atomic(SUPPORT_LOCK_PATH, {
        'identity': support_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })

DIRECTION_RUNTIME_DIR = Path('/content/visualtorque_semantic_direction_images_v1')
TARGET_RUNTIME_DIR = Path('/content/visualtorque_semantic_error_targets_v1')
DIRECTION_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
TARGET_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
direction_files = sorted({
    role['image_filename']
    for row in direction_support for role in row['roles'].values()
})
target_files = sorted({
    row['roles']['base']['image_filename'] for row in target_manifest
})
for filename in tqdm(direction_files, desc='Caching direction images', unit='image'):
    source = INVERSE_SCENE_DIR / filename
    destination = DIRECTION_RUNTIME_DIR / filename
    assert source.is_file(), source
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
for filename in tqdm(target_files, desc='Caching target images', unit='image'):
    source = TARGET_SCENE_DIR / filename
    destination = TARGET_RUNTIME_DIR / filename
    assert source.is_file(), source
    if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
print('Direction / target supports:', len(direction_support), len(target_manifest))
print('Geometry overlap:', len(set(target_signatures) & direction_signatures))


In [ ]:
%%cell_progress 7/17 Load the pinned model and mapped decoder layers
cell_stage(f'Loading {MODEL_KEY} weights for semantic error mediation')
import gc
if 'adapter' in globals():
    del adapter
    gc.collect()
    torch.cuda.empty_cache()

adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID, revision=MODEL_REVISION,
    dtype=MODEL_DTYPE, device_map=DEVICE_MAP,
)
display(adapter.metadata)

def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == EXPECTED_LAYER_COUNT
assert all(0 <= layer < len(DECODER_LAYERS) for layer in CAPTURE_LAYERS)
print('Layer path:', LAYER_PATH)
print('Capture layers:', CAPTURE_LAYERS)


## 3. L49 semantic control and L55 code mediation


In [ ]:
%%cell_progress 8/17 Define direction and counterbalanced target prompts
cell_stage('Separating physical semantics from A/B/C output codes')
import math
from visual_torque_toolkit.behavior import PromptCondition
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {
    'left': 'left side', 'balanced': 'equal', 'right': 'right side',
}

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    features = row['roles']['base']['features']
    exact = str(features['outcome'])
    shortcut = str(features['strategy_predictions'][
        row['shortcut_type'] + '_only'
    ])
    assert {exact, shortcut} == {'left', 'right'}
    balanced_label = LABELS[
        int(stable_key(f"balanced:{row['family_id']}")[:8], 16) % 3
    ]
    side_labels = [label for label in LABELS if label != balanced_label]
    mappings = {}
    for variant, exact_label, shortcut_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        semantic_by_label = {
            exact_label: exact, shortcut_label: shortcut,
            balanced_label: 'balanced',
        }
        mappings[variant] = mapping_by_label(tuple(
            semantic_by_label[label] for label in LABELS
        ))
    return mappings

def semantics_by_label(mapping):
    return tuple(
        next(item['semantic'] for item in mapping if item['label'] == label)
        for label in LABELS
    )

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}"
        for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT,
        'Question: ' + QUESTION_TEXT[task],
        'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option '
        'letter and nothing else. Valid replies are: A, B, C.',
    ])

def make_example(row, role_record, task, mapping, name, image_root, split):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"semantic_error:{row['family_id']}:{name}",
        question_id=f"semantic_error_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']),
        condition=PromptCondition.IMAGE_RULE,
        question_kind=task,
        variant='semantic_error_mediation', split=split,
        problem_type='semantic_error_label_swap',
        outcome=gold, net_torque=int(features['net_torque']),
        prompt=custom_prompt(task, mapping),
        image_path=(image_root / role_record['image_filename']).resolve(),
        labels=LABELS,
        option_text=tuple(SEMANTIC_TEXT[value] for value in semantics),
        option_semantics=semantics,
        gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for donor_task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{donor_task}__{role}'
            examples[key] = make_example(
                row, row['roles'][role], donor_task, mapping,
                key, DIRECTION_RUNTIME_DIR, 'direction_support',
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    return {
        variant: make_example(
            row, row['roles']['base'], 'outcome', mapping,
            f'target__{variant}', TARGET_RUNTIME_DIR, 'independent_target',
        )
        for variant, mapping in mappings.items()
    }, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples],
            add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {
        'examples': examples,
        'inputs': inputs,
        'token_by_label': token_by_label,
    }

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Direction and target prompt builders ready.')


In [ ]:
%%cell_progress 9/17 Define semantic steering and downstream capture scorers
cell_stage('Defining variable-layer final-token transport')

def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def forward_capture(prepared, layers):
    hidden_by_layer = {}
    handles = []
    for layer in layers:
        def hook(_module, _inputs, output, layer=layer):
            hidden_by_layer[layer] = output_hidden_tensor(
                output
            ).detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        for handle in handles:
            handle.remove()
    return score_records(prepared, output.logits), hidden_by_layer

def capture_examples(examples, layers):
    scores = {}
    hidden = {name: {} for name in examples}
    masks = {}
    for name, example in examples.items():
        prepared = prepare_examples([example])
        masks[name] = final_mask(prepared)
        current_scores, current_hidden = forward_capture(prepared, layers)
        scores[name] = current_scores[0]
        for layer, tensor in current_hidden.items():
            hidden[name][layer] = tensor[0]
        del prepared, current_hidden
    return scores, hidden, masks

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def forward_intervention_chunk(prepared, conditions):
    assert len(prepared['examples']) == len(conditions)
    layers = {int(row['layer_index']) for row in conditions}
    assert len(layers) == 1
    layer = next(iter(layers))
    payloads = torch.stack([row['payload_selected'] for row in conditions])
    masks = torch.stack([row['target_mask'] for row in conditions])
    modes = [row['intervention_mode'] for row in conditions]
    counts = []
    def hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payloads.to(device=hidden.device, dtype=hidden.dtype)
        mask = masks.to(device=hidden.device)
        modified = hidden.clone()
        counts.clear()
        for index, mode in enumerate(modes):
            current = mask[index]
            assert int(current.sum()) == int(payload[index].shape[0]) == 1
            if mode == 'add':
                modified[index, current] = hidden[index, current] + payload[index]
            elif mode == 'replace':
                modified[index, current] = payload[index]
            else:
                raise ValueError(mode)
            counts.append(int(current.sum()))
        return replace_output_hidden(output, modified)
    handle = DECODER_LAYERS[layer].register_forward_hook(hook)
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        handle.remove()
    records = score_records(prepared, output.logits)
    for record, condition, count in zip(records, conditions, counts):
        record.update({
            key: value for key, value in condition.items()
            if key not in ('payload_selected', 'target_mask')
        })
        record['patched_tokens'] = count
    return records

def score_conditions(example, conditions):
    records = []
    grouped = defaultdict(list)
    for condition in conditions:
        grouped[int(condition['layer_index'])].append(condition)
    for layer_conditions in grouped.values():
        for start in range(0, len(layer_conditions), PATCH_CONDITION_BATCH_SIZE):
            actual = layer_conditions[start:start + PATCH_CONDITION_BATCH_SIZE]
            padded = list(actual)
            while len(padded) < PATCH_CONDITION_BATCH_SIZE:
                padded.append(padded[-1])
            prepared = prepare_examples([example] * len(padded))
            records.extend(forward_intervention_chunk(
                prepared, padded
            )[:len(actual)])
            del prepared
    return records

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def label_score(record, label):
    labels = list(record['candidate_labels'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    index = labels.index(label)
    other = np.delete(values, index)
    return float(values[index] - np.logaddexp.reduce(other))

def entropy(record):
    probabilities = np.asarray(record['candidate_probabilities'], dtype=float)
    return float(-np.sum(probabilities * np.log(np.clip(probabilities, 1e-12, 1))))

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def score_l49_and_capture_l55(example, payload_selected, clean_l55, metadata):
    prepared = prepare_examples([example])
    mask = final_mask(prepared)
    captured = {}
    def semantic_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        payload = payload_selected.to(device=hidden.device, dtype=hidden.dtype)
        current = mask.to(hidden.device)
        modified = hidden.clone()
        modified[0, current] = hidden[0, current] + payload
        return replace_output_hidden(output, modified)
    def label_capture_hook(_module, _inputs, output):
        captured['hidden'] = output_hidden_tensor(output).detach().cpu().clone()
    semantic_handle = DECODER_LAYERS[SEMANTIC_LAYER].register_forward_hook(
        semantic_hook
    )
    label_handle = DECODER_LAYERS[LABEL_LAYER].register_forward_hook(
        label_capture_hook
    )
    try:
        with torch.inference_mode():
            output = adapter.model(**prepared['inputs'], use_cache=False)
    finally:
        semantic_handle.remove()
        label_handle.remove()
    record = score_records(prepared, output.logits)[0]
    downstream = captured['hidden'][0][mask] - clean_l55
    record.update(metadata)
    record['patched_tokens'] = int(mask.sum())
    record['downstream_l55_norm'] = selected_norm(downstream)
    return record, downstream

print('Semantic steering and downstream capture scorers ready.')


In [ ]:
%%cell_progress 10/17 Build frozen L49 semantic and L55 code libraries
cell_stage('Learning directions only from the disjoint depth support')

left_units = []
left_raw_norms = []
code_units = defaultdict(list)
code_raw_norms = []
direction_progress = tqdm(
    direction_support, desc='Building frozen direction libraries', unit='family'
)
for row in direction_progress:
    examples, mapping = build_direction_examples(row)
    scores, hidden, masks = capture_examples(examples, CAPTURE_LAYERS)
    for donor_task in DONOR_TASKS:
        base_key = f'{donor_task}__base'
        source_key = f'{donor_task}__aligned_source'
        base_semantic = task_semantic(
            row['roles']['base']['features'], donor_task
        )
        source_semantic = task_semantic(
            row['roles']['aligned_source']['features'], donor_task
        )
        assert {base_semantic, source_semantic} == {'left', 'right'}
        semantic_delta = (
            hidden[source_key][SEMANTIC_LAYER][masks[source_key]]
            - hidden[base_key][SEMANTIC_LAYER][masks[base_key]]
        )
        left_delta = semantic_delta if source_semantic == 'left' else -semantic_delta
        left_units.append(unit_direction(left_delta))
        left_raw_norms.append(selected_norm(semantic_delta))

        code_delta = (
            hidden[source_key][LABEL_LAYER][masks[source_key]]
            - hidden[base_key][LABEL_LAYER][masks[base_key]]
        )
        base_label = label_for(mapping, base_semantic)
        source_label = label_for(mapping, source_semantic)
        assert base_label != source_label
        code_units[(base_label, source_label)].append(unit_direction(code_delta))
        code_raw_norms.append(selected_norm(code_delta))
    del hidden
    torch.cuda.empty_cache()

assert len(left_units) == EXPECTED_DIRECTION_FAMILIES * len(DONOR_TASKS)
assert set(code_units) == {
    (left, right) for left in LABELS for right in LABELS if left != right
}
assert min(len(values) for values in code_units.values()) >= 4
SEMANTIC_STEERING_NORM = float(np.median(left_raw_norms))
CODE_STEERING_NORM = float(np.median(code_raw_norms))
SEMANTIC_DIRECTION = rescale_direction(
    torch.stack(left_units).mean(dim=0), SEMANTIC_STEERING_NORM
)
CODE_DIRECTIONS = {
    transition: rescale_direction(
        torch.stack(values).mean(dim=0), CODE_STEERING_NORM
    )
    for transition, values in code_units.items()
}
direction_metadata = {
    'schema_version': 'semantic-code-direction-library-1.0',
    'direction_support_sha256': sha256_file(DEPTH_SUPPORT_PATH),
    'semantic_layer': SEMANTIC_LAYER,
    'label_layer': LABEL_LAYER,
    'semantic_direction': 'left-minus-right',
    'semantic_examples': len(left_units),
    'semantic_norm': SEMANTIC_STEERING_NORM,
    'code_norm': CODE_STEERING_NORM,
    'code_transition_counts': {
        f'{left}->{right}': len(values)
        for (left, right), values in code_units.items()
    },
}
write_json_atomic(SHARED_ROOT / 'direction_library.json', direction_metadata)

# Mandatory parity check at both frozen sites.
smoke_row = target_manifest[0]
smoke_examples, smoke_mappings = build_target_examples(smoke_row)
smoke_scores, smoke_hidden, smoke_masks = capture_examples(
    smoke_examples, CAPTURE_LAYERS
)
smoke_errors = {}
for layer in CAPTURE_LAYERS:
    for variant in MAPPING_VARIANTS:
        self_record = score_conditions(smoke_examples[variant], [{
            'condition': 'self_replay', 'layer_index': layer,
            'intervention_mode': 'replace',
            'payload_selected': smoke_hidden[variant][layer][smoke_masks[variant]],
            'target_mask': smoke_masks[variant],
        }])[0]
        error = float(np.max(np.abs(
            centered_scores(self_record) - centered_scores(smoke_scores[variant])
        )))
        assert error < SMOKE_CENTERED_LOGPROB_TOLERANCE, (layer, variant, error)
        smoke_errors[f'{layer}:{variant}'] = error
print('Direction library:', direction_metadata)
print('Self replay:', smoke_errors)
del smoke_hidden
torch.cuda.empty_cache()


In [ ]:
%%cell_progress 11/17 Capture independent semantic-error and mediation shards
cell_stage('Testing both label swaps at L49 and L55')

capture_identity = {
    'schema_version': 'qwen27b-semantic-error-capture-1.0',
    'protocol_identity': canonical_protocol_identity,
    'support_identity': support_identity,
    'direction_metadata': direction_metadata,
    'model_id': MODEL_ID,
    'resolved_revision': adapter.metadata['resolved_revision'],
    'interventions': list(INTERVENTIONS),
    'records_per_family': RECORDS_PER_FAMILY,
}
CAPTURE_SHA256 = 'sha256:' + hashlib.sha256(
    json.dumps(capture_identity, sort_keys=True).encode('utf-8')
).hexdigest()
config_path = SHARED_ROOT / 'capture_config.json'
if config_path.is_file():
    previous = json.loads(config_path.read_text(encoding='utf-8'))
    assert previous.get('identity') == capture_identity
else:
    write_json_atomic(config_path, {
        'identity': capture_identity,
        'capture_identity_sha256': CAPTURE_SHA256,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    })

completed_ids = set()
for path in sorted(SHARD_DIR.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    assert payload['capture_identity_sha256'] == CAPTURE_SHA256
    assert len(payload['patch_records']) == RECORDS_PER_FAMILY
    completed_ids.add(payload['family_id'])
print('Completed / total:', len(completed_ids), len(target_manifest))

for row in tqdm(target_manifest, desc='Semantic error mediation', unit='family'):
    family_id = str(row['family_id'])
    if family_id in completed_ids:
        continue
    examples, mappings = build_target_examples(row)
    baselines, clean_hidden, masks = capture_examples(examples, CAPTURE_LAYERS)
    features = row['roles']['base']['features']
    exact = str(features['outcome'])
    shortcut = str(features['strategy_predictions'][
        row['shortcut_type'] + '_only'
    ])
    assert {exact, shortcut} == {'left', 'right'}
    semantic_exact = (
        SEMANTIC_DIRECTION if exact == 'left' else -SEMANTIC_DIRECTION
    )
    semantic_shortcut = -semantic_exact
    random_base = matched_random(
        SEMANTIC_DIRECTION, f'l49:{family_id}:random'
    )
    random_exact = random_base if exact == 'left' else -random_base
    random_shortcut = -random_exact
    records = []
    for variant in MAPPING_VARIANTS:
        other_variant = 'reverse' if variant == 'forward' else 'forward'
        mapping = mappings[variant]
        other_mapping = mappings[other_variant]
        exact_label = label_for(mapping, exact)
        shortcut_label = label_for(mapping, shortcut)
        other_exact_label = label_for(other_mapping, exact)
        other_shortcut_label = label_for(other_mapping, shortcut)
        exact_transition = (shortcut_label, exact_label)
        shortcut_transition = (exact_label, shortcut_label)
        stale_exact_transition = (other_shortcut_label, other_exact_label)
        stale_shortcut_transition = (other_exact_label, other_shortcut_label)
        code_exact = CODE_DIRECTIONS[exact_transition]
        code_shortcut = CODE_DIRECTIONS[shortcut_transition]
        stale_exact = CODE_DIRECTIONS[stale_exact_transition]
        stale_shortcut = CODE_DIRECTIONS[stale_shortcut_transition]
        random_code_exact = matched_random(
            code_exact, f'l55:{family_id}:{variant}:exact'
        )
        random_code_shortcut = matched_random(
            code_shortcut, f'l55:{family_id}:{variant}:shortcut'
        )
        clean_l55 = clean_hidden[variant][LABEL_LAYER][masks[variant]]
        common = {
            'family_id': family_id,
            'diagnostic_cell': row['diagnostic_cell'],
            'shortcut_type': row['shortcut_type'],
            'mapping_variant': variant,
            'exact_semantic': exact, 'shortcut_semantic': shortcut,
            'exact_label': exact_label, 'shortcut_label': shortcut_label,
        }
        for intervention, payload, correct_code, stale_code in (
            ('l49_semantic_exact', semantic_exact, code_exact, stale_exact),
            ('l49_semantic_shortcut', semantic_shortcut, code_shortcut, stale_shortcut),
            ('l49_random_exact', random_exact, code_exact, stale_exact),
            ('l49_random_shortcut', random_shortcut, code_shortcut, stale_shortcut),
        ):
            record, downstream = score_l49_and_capture_l55(
                examples[variant], payload, clean_l55,
                {**common, 'intervention': intervention,
                 'site_label': 'semantic_l49',
                 'layer_index': SEMANTIC_LAYER,
                 'payload_norm': selected_norm(payload)},
            )
            record['downstream_correct_code_cosine'] = cosine_similarity(
                downstream, correct_code
            )
            record['downstream_stale_code_cosine'] = cosine_similarity(
                downstream, stale_code
            )
            record['downstream_code_advantage'] = float(
                record['downstream_correct_code_cosine']
                - record['downstream_stale_code_cosine']
            )
            records.append(record)
        l55_conditions = []
        for intervention, payload in (
            ('l55_code_exact', code_exact),
            ('l55_code_shortcut', code_shortcut),
            ('l55_stale_exact', stale_exact),
            ('l55_stale_shortcut', stale_shortcut),
            ('l55_random_exact', random_code_exact),
            ('l55_random_shortcut', random_code_shortcut),
        ):
            l55_conditions.append({
                **common, 'intervention': intervention,
                'site_label': 'label_l55', 'layer_index': LABEL_LAYER,
                'intervention_mode': 'add',
                'payload_selected': payload,
                'target_mask': masks[variant],
                'payload_norm': selected_norm(payload),
            })
        records.extend(score_conditions(examples[variant], l55_conditions))
    assert len(records) == RECORDS_PER_FAMILY
    write_json_atomic(SHARD_DIR / f'{family_id}.json', {
        'schema_version': 'qwen27b-semantic-error-family-shard-1.0',
        'capture_identity_sha256': CAPTURE_SHA256,
        'family_id': family_id,
        'family': row,
        'baselines': baselines,
        'mappings': {
            key: list(value) for key, value in mappings.items()
        },
        'patch_records': records,
    })
    del clean_hidden
    torch.cuda.empty_cache()

shards = sorted(SHARD_DIR.glob('*.json'))
assert len(shards) == EXPECTED_TARGET_FAMILIES
assert sum(len(json.loads(path.read_text(encoding='utf-8'))[
    'patch_records'
]) for path in shards) == EXPECTED_RECORDS
print('Capture complete:', len(shards), 'families,', EXPECTED_RECORDS, 'records')


## 4. Bidirectional error tests and frozen result


In [ ]:
%%cell_progress 12/17 Release GPU and assemble family estimands
cell_stage('Grouping stable shortcut errors and stable exact-correct targets')
import gc
del adapter
gc.collect()
torch.cuda.empty_cache()

def semantic_logprob(record, semantic):
    semantics = list(record['candidate_semantics'])
    return float(record['candidate_logprobs'][semantics.index(semantic)])

def lookup(records, variant, intervention):
    matches = [
        record for record in records
        if record['mapping_variant'] == variant
        and record['intervention'] == intervention
    ]
    assert len(matches) == 1, (variant, intervention, len(matches))
    return matches[0]

variant_rows = []
family_rows = []
for path in tqdm(sorted(SHARD_DIR.glob('*.json')), desc='Estimands', unit='family'):
    payload = json.loads(path.read_text(encoding='utf-8'))
    family = payload['family']
    records = payload['patch_records']
    exact = str(family['roles']['base']['features']['outcome'])
    shortcut = str(family['roles']['base']['features']['strategy_predictions'][
        family['shortcut_type'] + '_only'
    ])
    baseline_predictions = {
        variant: str(payload['baselines'][variant]['predicted_semantic'])
        for variant in MAPPING_VARIANTS
    }
    if all(value == shortcut for value in baseline_predictions.values()):
        baseline_group = 'stable_shortcut_error'
    elif all(value == exact for value in baseline_predictions.values()):
        baseline_group = 'stable_exact_correct'
    else:
        baseline_group = 'mapping_sensitive_or_other'
    current_family_rows = []
    for variant in MAPPING_VARIANTS:
        clean = payload['baselines'][variant]
        clean_margin = semantic_logprob(clean, exact) - semantic_logprob(clean, shortcut)
        row = {
            'family_id': family['family_id'],
            'diagnostic_cell': family['diagnostic_cell'],
            'shortcut_type': family['shortcut_type'],
            'mapping_variant': variant,
            'exact_semantic': exact, 'shortcut_semantic': shortcut,
            'baseline_prediction': baseline_predictions[variant],
            'baseline_group': baseline_group,
            'clean_exact_shortcut_margin': clean_margin,
        }
        shifts = {}
        for intervention in INTERVENTIONS:
            record = lookup(records, variant, intervention)
            margin = (
                semantic_logprob(record, exact)
                - semantic_logprob(record, shortcut)
            )
            shifts[intervention] = margin - clean_margin
            row[f'{intervention}__shift'] = shifts[intervention]
            row[f'{intervention}__prediction'] = record['predicted_semantic']
        row.update({
            'l49_rescue_main': shifts['l49_semantic_exact'],
            'l49_rescue_specific': (
                shifts['l49_semantic_exact'] - shifts['l49_random_exact']
            ),
            'l49_induction_main': -shifts['l49_semantic_shortcut'],
            'l49_induction_specific': (
                shifts['l49_random_shortcut']
                - shifts['l49_semantic_shortcut']
            ),
            'l55_rescue_code_control': (
                shifts['l55_code_exact'] - shifts['l55_stale_exact']
            ),
            'l55_induction_code_control': (
                shifts['l55_stale_shortcut'] - shifts['l55_code_shortcut']
            ),
            'l55_rescue_vs_random': (
                shifts['l55_code_exact'] - shifts['l55_random_exact']
            ),
            'l55_induction_vs_random': (
                shifts['l55_random_shortcut'] - shifts['l55_code_shortcut']
            ),
            'mediation_exact': lookup(
                records, variant, 'l49_semantic_exact'
            )['downstream_code_advantage'],
            'mediation_shortcut': lookup(
                records, variant, 'l49_semantic_shortcut'
            )['downstream_code_advantage'],
        })
        variant_rows.append(row)
        current_family_rows.append(row)
    frame = pd.DataFrame(current_family_rows)
    aggregate = {
        key: float(frame[key].mean())
        for key in (
            'l49_rescue_main', 'l49_rescue_specific',
            'l49_induction_main', 'l49_induction_specific',
            'l55_rescue_code_control', 'l55_induction_code_control',
            'l55_rescue_vs_random', 'l55_induction_vs_random',
            'mediation_exact', 'mediation_shortcut',
        )
    }
    family_rows.append({
        'family_id': family['family_id'],
        'diagnostic_cell': family['diagnostic_cell'],
        'shortcut_type': family['shortcut_type'],
        'exact_semantic': exact, 'shortcut_semantic': shortcut,
        'baseline_group': baseline_group,
        'forward_prediction': baseline_predictions['forward'],
        'reverse_prediction': baseline_predictions['reverse'],
        **aggregate,
    })

variant_frame = pd.DataFrame(variant_rows)
family_frame = pd.DataFrame(family_rows)
variant_frame.to_csv(ANALYSIS_DIR / 'variant_estimands.csv', index=False)
family_frame.to_csv(ANALYSIS_DIR / 'family_estimands.csv', index=False)
cohort_counts = family_frame.groupby([
    'diagnostic_cell', 'baseline_group'
]).size().rename('n').reset_index()
display(cohort_counts)
print('Family / variant estimands:', len(family_frame), len(variant_frame))


In [ ]:
%%cell_progress 13/17 Test bidirectional error control and L49-to-L55 mediation
cell_stage('Bootstrapping frozen pooled and mapping-specific estimands')

def bootstrap_summary(values, key):
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    result = {
        'n': int(len(array)),
        'mean': float(array.mean()) if len(array) else np.nan,
        'ci_low': np.nan, 'ci_high': np.nan, 'p_directional': np.nan,
    }
    if len(array) < 2:
        return result
    generator = np.random.default_rng(
        int(stable_key(f'bootstrap:{key}')[:16], 16) % (2**63 - 1)
    )
    draws = generator.choice(
        array, size=(BOOTSTRAP_REPLICATES, len(array)), replace=True
    ).mean(axis=1)
    result.update({
        'ci_low': float(np.quantile(draws, ALPHA / 2)),
        'ci_high': float(np.quantile(draws, 1 - ALPHA / 2)),
        'p_directional': float(
            (1 + np.count_nonzero(draws <= 0))
            / (BOOTSTRAP_REPLICATES + 1)
        ),
    })
    return result

def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full_like(values, np.nan)
    finite = np.flatnonzero(np.isfinite(values))
    if not len(finite):
        return adjusted
    finite_values = values[finite]
    order = np.argsort(finite_values)
    running = 1.0
    for reverse_rank in range(len(finite_values) - 1, -1, -1):
        local_index = order[reverse_rank]
        index = finite[local_index]
        rank = reverse_rank + 1
        running = min(
            running,
            finite_values[local_index] * len(finite_values) / rank,
        )
        adjusted[index] = running
    return adjusted

rescue = family_frame[
    family_frame['baseline_group'] == 'stable_shortcut_error'
]
induction = family_frame[
    family_frame['baseline_group'] == 'stable_exact_correct'
]
primary_vectors = {
    'rescue_main': rescue['l49_rescue_main'],
    'rescue_specific': rescue['l49_rescue_specific'],
    'induction_main': induction['l49_induction_main'],
    'induction_specific': induction['l49_induction_specific'],
    'mediation_exact': family_frame['mediation_exact'],
    'mediation_shortcut': family_frame['mediation_shortcut'],
    'code_rescue_control': family_frame['l55_rescue_code_control'],
    'code_induction_control': family_frame['l55_induction_code_control'],
    'code_rescue_vs_random': family_frame['l55_rescue_vs_random'],
    'code_induction_vs_random': family_frame['l55_induction_vs_random'],
}
test_rows = []
for metric, values in primary_vectors.items():
    test_rows.append({
        'test_family': 'primary', 'diagnostic_cell': 'pooled',
        'mapping_variant': 'averaged', 'metric': metric,
        'primary_test': True,
        **bootstrap_summary(values, f'primary:{metric}'),
    })

# Mapping invariance is mandatory but not added to the multiplicity family.
for variant in MAPPING_VARIANTS:
    current = variant_frame[variant_frame['mapping_variant'] == variant]
    for direction, group_name, metric in (
        ('rescue', 'stable_shortcut_error', 'l49_rescue_specific'),
        ('induction', 'stable_exact_correct', 'l49_induction_specific'),
    ):
        values = current[current['baseline_group'] == group_name][metric]
        test_rows.append({
            'test_family': 'mapping_invariance',
            'diagnostic_cell': 'pooled', 'mapping_variant': variant,
            'metric': direction, 'primary_test': False,
            **bootstrap_summary(values, f'mapping:{variant}:{direction}'),
        })

# Conflict-specific estimates are secondary and descriptive.
for diagnostic_cell in CONFLICT_CELLS:
    for direction, group_name, metric in (
        ('rescue', 'stable_shortcut_error', 'l49_rescue_specific'),
        ('induction', 'stable_exact_correct', 'l49_induction_specific'),
    ):
        values = family_frame[
            (family_frame['diagnostic_cell'] == diagnostic_cell)
            & (family_frame['baseline_group'] == group_name)
        ][metric]
        test_rows.append({
            'test_family': 'conflict_secondary',
            'diagnostic_cell': diagnostic_cell,
            'mapping_variant': 'averaged', 'metric': direction,
            'primary_test': False,
            **bootstrap_summary(values, f'cell:{diagnostic_cell}:{direction}'),
        })

tests_frame = pd.DataFrame(test_rows)
tests_frame['q_value'] = np.nan
primary_index = tests_frame.index[tests_frame['primary_test']]
tests_frame.loc[primary_index, 'q_value'] = bh_adjust(
    tests_frame.loc[primary_index, 'p_directional'].to_numpy(float)
)

def primary_test(metric):
    rows = tests_frame[
        (tests_frame['test_family'] == 'primary')
        & (tests_frame['metric'] == metric)
    ]
    assert len(rows) == 1
    return rows.iloc[0]

def mapping_test(variant, direction):
    rows = tests_frame[
        (tests_frame['test_family'] == 'mapping_invariance')
        & (tests_frame['mapping_variant'] == variant)
        & (tests_frame['metric'] == direction)
    ]
    assert len(rows) == 1
    return rows.iloc[0]

rescue_main = primary_test('rescue_main')
rescue_specific = primary_test('rescue_specific')
induction_main = primary_test('induction_main')
induction_specific = primary_test('induction_specific')
rescue_passed = bool(
    rescue_main['n'] >= MIN_COHORT
    and rescue_main['ci_low'] > ERROR_SESOI_LOGODDS
    and rescue_specific['ci_low'] > 0
    and rescue_main['q_value'] < ALPHA
    and rescue_specific['q_value'] < ALPHA
    and all(mapping_test(variant, 'rescue')['ci_low'] > 0
            for variant in MAPPING_VARIANTS)
)
induction_passed = bool(
    induction_main['n'] >= MIN_COHORT
    and induction_main['ci_low'] > ERROR_SESOI_LOGODDS
    and induction_specific['ci_low'] > 0
    and induction_main['q_value'] < ALPHA
    and induction_specific['q_value'] < ALPHA
    and all(mapping_test(variant, 'induction')['ci_low'] > 0
            for variant in MAPPING_VARIANTS)
)
mediation_passed = bool(all(
    primary_test(metric)['ci_low'] > MEDIATION_SESOI_COSINE
    and primary_test(metric)['q_value'] < ALPHA
    for metric in ('mediation_exact', 'mediation_shortcut')
))
code_control_passed = bool(all(
    primary_test(metric)['ci_low'] > 0
    and primary_test(metric)['q_value'] < ALPHA
    for metric in (
        'code_rescue_control', 'code_induction_control',
        'code_rescue_vs_random', 'code_induction_vs_random',
    )
))

if rescue_passed and induction_passed and mediation_passed and code_control_passed:
    primary_decision = 'bidirectional_semantic_error_control_with_code_mediation'
elif rescue_passed and induction_passed:
    primary_decision = 'bidirectional_semantic_error_control_without_full_mediation'
elif rescue_passed or induction_passed:
    primary_decision = 'directional_semantic_error_control_only'
elif rescue_main['n'] < MIN_COHORT or induction_main['n'] < MIN_COHORT:
    primary_decision = 'stable_mapping_cohorts_underpowered'
else:
    primary_decision = 'no_confirmed_semantic_error_control'

decision_payload = {
    'schema_version': 'qwen27b-semantic-error-mediation-decision-1.0',
    'model_id': MODEL_ID, 'resolved_revision': MODEL_REVISION,
    'semantic_layer': SEMANTIC_LAYER, 'label_layer': LABEL_LAYER,
    'primary_decision': primary_decision,
    'rescue_passed': rescue_passed,
    'induction_passed': induction_passed,
    'mediation_passed': mediation_passed,
    'code_control_passed': code_control_passed,
    'rescue_n': int(rescue_main['n']),
    'induction_n': int(induction_main['n']),
    'primary_metrics': {
        metric: json.loads(primary_test(metric).to_json())
        for metric in primary_vectors
    },
}
tests_frame.to_csv(ANALYSIS_DIR / 'semantic_error_tests.csv', index=False)
write_json_atomic(ANALYSIS_DIR / 'semantic_error_decision.json', decision_payload)
display(tests_frame.round(4))
display(pd.DataFrame([decision_payload]).drop(columns=['primary_metrics']))


In [ ]:
%%cell_progress 14/17 Audit independence, cohorts, mappings, norms, and parity
cell_stage('Checking all frozen confirmatory invariants')
mapping_violations = []
for row in target_manifest:
    mappings = target_mappings(row)
    exact = row['roles']['base']['features']['outcome']
    shortcut = row['roles']['base']['features']['strategy_predictions'][
        row['shortcut_type'] + '_only'
    ]
    forward = mappings['forward']
    reverse = mappings['reverse']
    if not (
        label_for(forward, exact) == label_for(reverse, shortcut)
        and label_for(forward, shortcut) == label_for(reverse, exact)
        and label_for(forward, 'balanced') == label_for(reverse, 'balanced')
    ):
        mapping_violations.append(row['family_id'])
assert not mapping_violations

record_count = 0
semantic_norm_errors = []
code_norm_errors = []
for path in sorted(SHARD_DIR.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    records = payload['patch_records']
    record_count += len(records)
    for record in records:
        norm = float(record['payload_norm'])
        if record['site_label'] == 'semantic_l49':
            semantic_norm_errors.append(
                abs(norm - SEMANTIC_STEERING_NORM) / SEMANTIC_STEERING_NORM
            )
        else:
            code_norm_errors.append(
                abs(norm - CODE_STEERING_NORM) / CODE_STEERING_NORM
            )
assert record_count == EXPECTED_RECORDS
maximum_semantic_norm_error = float(max(semantic_norm_errors))
maximum_code_norm_error = float(max(code_norm_errors))
assert maximum_semantic_norm_error < NORM_RELATIVE_TOLERANCE
assert maximum_code_norm_error < NORM_RELATIVE_TOLERANCE

cohort_payload = {
    str(row['diagnostic_cell']) + ':' + str(row['baseline_group']): int(row['n'])
    for _, row in cohort_counts.iterrows()
}
audit_payload = {
    'schema_version': 'qwen27b-semantic-error-mediation-audit-1.0',
    'support_identity': support_identity,
    'direction_target_geometry_overlap': 0,
    'physics_violations': len(physics_violations),
    'mapping_violations': len(mapping_violations),
    'target_families': len(target_manifest),
    'records': record_count,
    'cohort_counts': cohort_payload,
    'max_self_replay_error': float(max(smoke_errors.values())),
    'self_replay_tolerance': SMOKE_CENTERED_LOGPROB_TOLERANCE,
    'maximum_semantic_norm_error': maximum_semantic_norm_error,
    'maximum_code_norm_error': maximum_code_norm_error,
    'norm_tolerance': NORM_RELATIVE_TOLERANCE,
}
write_json_atomic(ANALYSIS_DIR / 'robustness_audit.json', audit_payload)
display(pd.DataFrame([audit_payload]))


In [ ]:
%%cell_progress 15/17 Save the paper-facing semantic-error figure
cell_stage('Rendering behavior cohorts, bidirectional control, and mediation')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

cohort_plot = cohort_counts.pivot(
    index='diagnostic_cell', columns='baseline_group', values='n'
).fillna(0)
cohort_plot.plot(kind='bar', ax=axes[0, 0], color=['#9CA3AF', '#16A34A', '#DC2626'])
axes[0, 0].set_title('A. Stable label-remapped behavior cohorts')
axes[0, 0].set_ylabel('Independent families')
axes[0, 0].tick_params(axis='x', rotation=20)
axes[0, 0].legend(frameon=False, fontsize=8)

error_metrics = ['rescue_main', 'induction_main']
error_rows = [primary_test(metric) for metric in error_metrics]
means = np.asarray([row['mean'] for row in error_rows], float)
errors = np.vstack([
    means - np.asarray([row['ci_low'] for row in error_rows], float),
    np.asarray([row['ci_high'] for row in error_rows], float) - means,
])
axes[0, 1].bar(['Rescue', 'Induction'], means, color=['#16A34A', '#DC2626'])
axes[0, 1].errorbar(range(2), means, yerr=errors, fmt='none', color='black', capsize=4)
axes[0, 1].axhline(ERROR_SESOI_LOGODDS, color='#2563EB', linestyle='--')
axes[0, 1].set_title('B. L49 bidirectional semantic control')
axes[0, 1].set_ylabel('Exact-direction log-odds effect')

mapping_means = np.asarray([
    [mapping_test(variant, direction)['mean'] for direction in ('rescue', 'induction')]
    for variant in MAPPING_VARIANTS
])
x = np.arange(2)
axes[1, 0].bar(x - 0.18, mapping_means[0], 0.36, label='Forward', color='#2563EB')
axes[1, 0].bar(x + 0.18, mapping_means[1], 0.36, label='Reverse', color='#D97706')
axes[1, 0].axhline(0, color='black', linewidth=1)
axes[1, 0].set_xticks(x, ['Rescue', 'Induction'])
axes[1, 0].set_title('C. Invariance to swapped A/B/C labels')
axes[1, 0].legend(frameon=False)

mediation_metrics = [
    'mediation_exact', 'mediation_shortcut',
    'code_rescue_control', 'code_induction_control',
]
mediation_rows = [primary_test(metric) for metric in mediation_metrics]
mediation_means = np.asarray([row['mean'] for row in mediation_rows], float)
mediation_errors = np.vstack([
    mediation_means - np.asarray([row['ci_low'] for row in mediation_rows], float),
    np.asarray([row['ci_high'] for row in mediation_rows], float) - mediation_means,
])
axes[1, 1].bar(
    ['L49→L55\nexact', 'L49→L55\nshortcut', 'L55 code\nexact', 'L55 code\nshortcut'],
    mediation_means, color=['#7C3AED', '#7C3AED', '#D97706', '#D97706'],
)
axes[1, 1].errorbar(
    range(4), mediation_means, yerr=mediation_errors,
    fmt='none', color='black', capsize=4,
)
axes[1, 1].axhline(0, color='black', linewidth=1)
axes[1, 1].set_title('D. Target-specific downstream code conversion')
axes[1, 1].tick_params(axis='x', rotation=15)

fig.suptitle('VisualTorque: Semantic Error Control and Code Mediation', fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.96))
figure_path = FIGURE_DIR / 'semantic_error_mediation.png'
fig.savefig(figure_path, dpi=220, bbox_inches='tight')
plt.show()
print('Figure:', figure_path)


In [ ]:
%%cell_progress 16/17 Write the frozen result card
cell_stage('Writing the strongest supported mechanism statement')
interpretations = {
    'bidirectional_semantic_error_control_with_code_mediation': (
        'The frozen L49 physical-side semantic both repairs stable shortcut '
        'errors and induces the matching error under swapped output labels. '
        'Its downstream L55 change aligns with the target prompt-specific '
        'code, completing a response-semantic-to-output-code causal account.'
    ),
    'bidirectional_semantic_error_control_without_full_mediation': (
        'L49 has bidirectional label-invariant error control, but the frozen '
        'L49-to-L55 code-alignment gate does not fully pass.'
    ),
    'directional_semantic_error_control_only': (
        'L49 controls only one behavioral direction. It is a causal bias but '
        'not a sufficient bidirectional error mechanism.'
    ),
    'stable_mapping_cohorts_underpowered': (
        'Too few families retain the same exact or shortcut prediction under '
        'both label mappings for the predeclared bidirectional claim.'
    ),
    'no_confirmed_semantic_error_control': (
        'The frozen L49 semantic does not pass the label-invariant error-control '
        'gates on the independent conflict targets.'
    ),
}
primary_decision = decision_payload['primary_decision']
decision_payload['interpretation'] = interpretations[primary_decision]
decision_payload['audit'] = audit_payload
decision_payload['support_identity'] = support_identity
decision_payload['limitations'] = [
    protocol_identity['interpretation_limit'],
    'L49 and L55 were frozen from one Qwen-27B depth study; no all-model claim is made.',
    'The targets are independent of site discovery but were used by an earlier experiment at different sites.',
    'Stable behavior cohorts are defined only after the model-blind target manifest is locked.',
    'Weight- and distance-conflict estimates are secondary; the powered primary test pools both error classes.',
]
decision_payload['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
write_json_atomic(ANALYSIS_DIR / 'semantic_error_decision.json', decision_payload)

lines = [
    '# Qwen-27B Semantic Error Control and Code Mediation', '',
    f'- Decision: **{primary_decision}**',
    f"- Rescue: `{decision_payload['rescue_passed']}` (n={decision_payload['rescue_n']})",
    f"- Induction: `{decision_payload['induction_passed']}` (n={decision_payload['induction_n']})",
    f"- L49→L55 mediation: `{decision_payload['mediation_passed']}`",
    f"- L55 code control: `{decision_payload['code_control_passed']}`", '',
    '## Interpretation', '', interpretations[primary_decision], '',
    '## Limits', '',
]
lines.extend(f'- {value}' for value in decision_payload['limitations'])
(ANALYSIS_DIR / 'REPORT.md').write_text(
    '\n'.join(lines) + '\n', encoding='utf-8'
)
display(decision_payload)
print('Report:', ANALYSIS_DIR / 'REPORT.md')


In [ ]:
%%cell_progress 17/17 Save the paper-facing mechanism summary
cell_stage('Exporting compact tables for manuscript integration')
summary_rows = []
for metric in (
    'rescue_main', 'rescue_specific', 'induction_main', 'induction_specific',
    'mediation_exact', 'mediation_shortcut',
    'code_rescue_control', 'code_induction_control',
    'code_rescue_vs_random', 'code_induction_vs_random',
):
    row = primary_test(metric)
    summary_rows.append({
        'metric': metric, 'n': int(row['n']), 'mean': float(row['mean']),
        'ci_low': float(row['ci_low']), 'ci_high': float(row['ci_high']),
        'q_value': float(row['q_value']),
    })
summary_frame = pd.DataFrame(summary_rows)
summary_frame.to_csv(ANALYSIS_DIR / 'paper_primary_estimands.csv', index=False)
secondary_frame = tests_frame[
    tests_frame['test_family'] == 'conflict_secondary'
].copy()
secondary_frame.to_csv(
    ANALYSIS_DIR / 'paper_conflict_secondary.csv', index=False
)
display(summary_frame.round(4))
display(secondary_frame.round(4))
print('Primary decision:', primary_decision)
print('Paper outputs:', ANALYSIS_DIR)
